# Ensemble interaction graph analysis

This notebook produces the **same interaction-graph analysis** independently for two ensembles:

| Ensemble | Read `.bnet` files from | Save results to |
|---|---|---|
| Our models | `../Results/Boolean_network_sampling_outputs/` | `../Results/Ensemble_analysis_outputs/Our_models/` |
| Their models | `../Data/models/` | `../Results/Ensemble_analysis_outputs/Their_models/` |

The notebooks' existing logic is retained: for each gene, count distinct Boolean functions; for each signed regulatory influence, count the models where it occurs; and draw the resulting influence graph.

The number of models is detected automatically for each ensemble. The two model collections are **never mixed**.

## 1. Set the input and output folders

These are the only paths you should need to change if you move the project. All generated files stay inside the two output subfolders.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import json

import mpbn
import pandas as pd
import graphviz

OUTPUT_ROOT = Path("../Results/Ensemble_analysis_outputs")
MODEL_FOLDERS = {
    "Our_models": Path("../Results/Boolean_network_sampling_outputs"),
    "Their_models": Path("../Data/models"),
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ipylab module is not installed, menus and toolbar are disabled.


## 2. Load the models and count their functions and regulatory influences

For each `.bnet` file:

- Each gene's Boolean function is recorded, so we can count **how many different rules** it has across the ensemble.
- Each activating or inhibiting influence is counted **once per model**.
- Constant functions (always 0 or always 1) are counted separately.

Nothing in this step combines the two ensembles.

In [2]:
def model_sort_key(path):
    # Order bn0, bn1, bn2, ... numerically, not alphabetically.
    name = path.stem
    if name.startswith("bn") and name[2:].isdigit():
        return (0, int(name[2:]))
    return (1, name)


def load_model_files(folder):
    if not folder.is_dir():
        raise FileNotFoundError(f"Model directory not found: {folder}")
    files = sorted(folder.glob("bn*.bnet"), key=model_sort_key)
    if not files:
        raise FileNotFoundError(f"No bn*.bnet files found in {folder}")
    return files


def analyze_models(folder):
    files = load_model_files(folder)
    all_functions = defaultdict(set)
    constant_counts = defaultdict(Counter)
    influence_counts = defaultdict(lambda: defaultdict(Counter))
    expected_genes = None

    for file in files:
        model = mpbn.MPBooleanNetwork(str(file))
        genes = set(model.keys())
        if expected_genes is None:
            expected_genes = genes
        elif genes != expected_genes:
            raise ValueError(f"Gene set differs in {file.name}; all models within an ensemble must use the same genes.")

        for target in model.keys():
            rule = model[target]
            all_functions[target].add(rule)

            if rule == True or rule == False:
                constant_counts[target][bool(rule)] += 1
                continue

            # Convert the logical function into clauses (same method as original notebook).
            clauses = mpbn.minibn.struct_of_dnf(model.ba, rule)
            seen_regulators = set()
            for conjunction in clauses:
                for regulator, sign in conjunction:
                    # Preserve original behavior: count a regulator once per target/model.
                    if regulator not in seen_regulators:
                        influence_counts[target][regulator][bool(sign)] += 1
                        seen_regulators.add(regulator)

    return {
        "files": files,
        "n_models": len(files),
        "genes": sorted(expected_genes),
        "functions": all_functions,
        "constants": constant_counts,
        "influences": influence_counts,
    }

## 3. Create the interaction graph

**Nodes:** genes. Their labels show the number of different Boolean functions across the ensemble. Dark yellow = one function; light yellow = two; beige = three. Rectangles indicate a gene assigned a constant function in at least one model.

**Arrows:** activating (green) or inhibitory (red) influences. The number on an arrow is **how many models contain it**. Influences present in **all** models are drawn thicker and darker.

The code uses each ensemble's own model count, rather than assuming 50 or 1,000.

In [3]:
def get_arcs_size(values, min_intensity=1, max_intensity=8):
    # Keep the original frequency-to-line-width mapping.
    result = {}

    def assign(parts, low, high, left=False):
        if not parts:
            return
        gaps = [parts[i + 1] - parts[i] for i in range(len(parts) - 1)]
        if not gaps or max(gaps) == 0 or low == high:
            for number in parts:
                result[number] = str(low if left else high)
            return
        split = gaps.index(max(gaps))
        middle = (low + high) // 2
        assign(parts[:split + 1], low, middle, True)
        assign(parts[split + 1:], middle + 1, high)

    assign(sorted(set(values)), min_intensity, max_intensity)
    return result


def create_influence_graph(stats):
    n_models = stats["n_models"]
    functions = stats["functions"]
    constants = stats["constants"]
    influences = stats["influences"]

    counts = [
        number
        for targets in influences.values()
        for signs in targets.values()
        for number in signs.values()
    ]
    widths = get_arcs_size(counts)
    widths[n_models] = "13"  # Highlight influences found in every model.

    dot = graphviz.Digraph("models_per_edge", comment="Number of models containing each regulatory influence")
    dot.graph_attr["ratio"] = "1.1"

    for gene in stats["genes"]:
        n_functions = len(functions[gene])
        if n_functions == 1:
            fill, style, fontsize = "darkgoldenrod2", "rounded,filled,bold", "50pt"
        elif n_functions == 2:
            fill, style, fontsize = "lightgoldenrod1", "rounded,filled", "40pt"
        elif n_functions == 3:
            fill, style, fontsize = "cornsilk", "rounded,filled", "35pt"
        elif n_functions < 10:
            fill, style, fontsize = "white", "rounded,filled", "30pt"
        else:
            fill, style, fontsize = "white", "rounded,filled,dotted", "30pt"

        label = f"{gene}\n{n_functions} func"
        if gene in constants:
            # Show both constant values if they occur among the models.
            for val in (False, True):
                if constants[gene][val]:
                    label += f"\n{int(val)}: {constants[gene][val]}"

        dot.node(
            gene, label=label, shape="box" if gene in constants else "oval",
            fillcolor=fill, style=style, fontsize=fontsize,
            fontcolor="black", fontname="arial bold",
        )

    for target, sources in influences.items():
        for source, signs in sources.items():
            for positive, number in signs.items():
                universal = number == n_models
                dot.edge(
                    source, target, label=str(number),
                    penwidth=widths[number],
                    color=("darkgreen" if universal else "darkolivegreen3") if positive else ("firebrick" if universal else "lightcoral"),
                    fontcolor=("darkgreen" if universal else "darkolivegreen3") if positive else ("firebrick" if universal else "lightcoral"),
                    arrowhead="normal" if positive else "tee",
                    fontname="arial bold", fontsize="30pt",
                    arrowsize="2" if positive else "3",
                )
    return dot

## 4. Save the results for each ensemble

Each output subfolder contains:

- **`IGstat.pdf`** — the original-style interaction graph.
- **`IGstat.gv`** — the editable Graphviz source.
- **`node_function_variability.csv`** — number of different Boolean functions per gene, plus constant-rule counts.
- **`edge_frequencies.csv`** — signed edges and the number and percentage of models containing each edge.
- **`edge_occurrence_summary.csv`** — how many edges occur in a given number of models.
- **`invariant_genes.csv`** — genes assigned the same Boolean function in every model.
- **`summary.json`** — model, gene and edge counts.

Both folders use identical analysis and output formats.

In [4]:
def save_ensemble_results(stats, output_folder):
    output_folder.mkdir(parents=True, exist_ok=True)
    n_models = stats["n_models"]

    gene_rows = []
    for gene in stats["genes"]:
        gene_rows.append({
            "gene": gene,
            "distinct_boolean_functions": len(stats["functions"][gene]),
            "constant_0_models": stats["constants"][gene][False],
            "constant_1_models": stats["constants"][gene][True],
        })
    gene_df = pd.DataFrame(gene_rows)
    gene_df.to_csv(output_folder / "node_function_variability.csv", index=False)
    gene_df.loc[gene_df["distinct_boolean_functions"] == 1].to_csv(
        output_folder / "invariant_genes.csv", index=False
    )

    edge_rows = []
    for target, sources in stats["influences"].items():
        for source, signs in sources.items():
            for positive, count in signs.items():
                edge_rows.append({
                    "source": source,
                    "target": target,
                    "influence": "activation" if positive else "inhibition",
                    "n_models": count,
                    "percent_models": round(100 * count / n_models, 2),
                })

    edge_columns = ["source", "target", "influence", "n_models", "percent_models"]
    edge_df = pd.DataFrame(edge_rows, columns=edge_columns)
    edge_df.to_csv(output_folder / "edge_frequencies.csv", index=False)

    # Save the count-by-frequency summary printed by the original notebook.
    frequencies = Counter(edge_df["n_models"])
    frequency_df = pd.DataFrame([
        {"n_models_with_edge": frequency, "n_edges": count}
        for frequency, count in sorted(frequencies.items())
    ], columns=["n_models_with_edge", "n_edges"])
    frequency_df.to_csv(output_folder / "edge_occurrence_summary.csv", index=False)

    dot = create_influence_graph(stats)
    # Keep both the .gv source and the PDF inside the requested output folder.
    dot.render(
        filename="IGstat.gv", directory=str(output_folder),
        outfile=str(output_folder / "IGstat.pdf"), format="pdf", view=False,
    )

    summary = {
        "model_count": n_models,
        "gene_count": len(stats["genes"]),
        "signed_edge_count": len(edge_df),
        "invariant_gene_count": int((gene_df["distinct_boolean_functions"] == 1).sum()),
    }
    (output_folder / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"\n{output_folder.name}: {n_models} models, {summary['gene_count']} genes")
    print(f"Invariant genes: {summary['invariant_gene_count']}")
    for row in frequency_df.itertuples(index=False):
        print(f"{row.n_edges:2d} edges present in {row.n_models_with_edge:4d} models")
    print(f"Saved to: {output_folder}")
    return summary

## 5. Run both analyses

This cell processes **our models** and **their models** separately. It reads all `bn*.bnet` files found in each folder. No files are written back into the input folders.

In [5]:
summaries = {}

for ensemble_name, input_folder in MODEL_FOLDERS.items():
    print(f"\nAnalyzing {ensemble_name} from {input_folder} ...")
    stats = analyze_models(input_folder)
    summaries[ensemble_name] = save_ensemble_results(
        stats, OUTPUT_ROOT / ensemble_name
    )

print("\nBoth analyses completed.")
display(pd.DataFrame(summaries).T)


Analyzing Our_models from ../Results/Boolean_network_sampling_outputs ...

Our_models: 50 models, 39 genes
Invariant genes: 4
 4 edges present in    1 models
 6 edges present in    2 models
10 edges present in    3 models
 4 edges present in    4 models
 7 edges present in    5 models
 7 edges present in    6 models
12 edges present in    7 models
10 edges present in    8 models
 7 edges present in    9 models
 9 edges present in   10 models
 8 edges present in   11 models
 5 edges present in   12 models
 3 edges present in   13 models
 5 edges present in   14 models
 7 edges present in   15 models
 4 edges present in   16 models
 2 edges present in   17 models
 2 edges present in   18 models
 2 edges present in   19 models
 3 edges present in   21 models
 1 edges present in   22 models
 1 edges present in   23 models
 2 edges present in   24 models
 2 edges present in   25 models
 3 edges present in   26 models
 3 edges present in   28 models
 2 edges present in   31 models
 3 edges 

,model_count,gene_count,signed_edge_count,invariant_gene_count
Our_models,50,39,145,4
Their_models,1000,39,124,12
